In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_excel('../raw_data/Online Retail.xlsx')

In [3]:
print(f"Rows loaded: {len(df):,}")

Rows loaded: 541,909


In [4]:
print(f"Columns: {df.shape[1]}")

Columns: 8


In [5]:
before = len(df)
df.drop_duplicates(inplace=True)
after = len(df)

print(f"Duplicates removed : {before - after:,}")
print(f"Rows remaining     : {after:,}")

Duplicates removed : 5,268
Rows remaining     : 536,641


In [6]:
print("Missing before:\n", df.isnull().sum())

Missing before:
 InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135037
Country             0
dtype: int64


In [7]:
# Drop rows where CustomerID is missing
# Reason: We need CustomerID for ALL customer-level analysis (CLV, retention, segmentation)
df.dropna(subset=['CustomerID'], inplace=True)

In [8]:
# Fill missing Description with 'Unknown'
df['Description'].fillna('Unknown', inplace=True)

C:\Users\Admin\AppData\Local\Temp\ipykernel_2144\2369539016.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['Description'].fillna('Unknown', inplace=True)


In [9]:
print(f"\nRows after dropping missing CustomerID: {len(df):,}")
print("Missing after:\n", df.isnull().sum())


Rows after dropping missing CustomerID: 401,604
Missing after:
 InvoiceNo      0
StockCode      0
Description    0
Quantity       0
InvoiceDate    0
UnitPrice      0
CustomerID     0
Country        0
dtype: int64


In [10]:
# Convert InvoiceDate to proper datetime format
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

In [11]:
# Convert CustomerID from float to integer (removes the .0)
df['CustomerID'] = df['CustomerID'].astype(int)

In [12]:
# Convert InvoiceNo to string (it's an ID, not a number)
df['InvoiceNo'] = df['InvoiceNo'].astype(str)

In [13]:
print("Updated data types:")
print(df.dtypes)
print(f"\nSample InvoiceDate: {df['InvoiceDate'].iloc[0]}")
print(f"Sample CustomerID : {df['CustomerID'].iloc[0]}")

Updated data types:
InvoiceNo              object
StockCode              object
Description            object
Quantity                int64
InvoiceDate    datetime64[ns]
UnitPrice             float64
CustomerID              int64
Country                object
dtype: object

Sample InvoiceDate: 2010-12-01 08:26:00
Sample CustomerID : 17850


In [14]:
# Remove cancelled invoices (InvoiceNo starting with 'C')
before = len(df)
df = df[~df['InvoiceNo'].str.startswith('C')]
print(f"Cancelled invoices removed: {before - len(df):,}")

Cancelled invoices removed: 8,872


In [15]:
# Remove rows with negative or zero Quantity (returns)
before = len(df)
df = df[df['Quantity'] > 0]
print(f"Negative quantity rows removed: {before - len(df):,}")

print(f"Rows remaining: {len(df):,}")

Negative quantity rows removed: 0
Rows remaining: 392,732


In [16]:
# Remove rows where UnitPrice is zero or negative
before = len(df)
df = df[df['UnitPrice'] > 0]
print(f"Zero/negative price rows removed: {before - len(df):,}")
print(f"Rows remaining: {len(df):,}")

Zero/negative price rows removed: 40
Rows remaining: 392,692


Feature engineering

In [17]:
# Recalculate TotalRevenue on clean data
df['TotalRevenue'] = df['Quantity'] * df['UnitPrice']

In [18]:
# Extract time-based features from InvoiceDate
df['Year']      = df['InvoiceDate'].dt.year
df['Month']     = df['InvoiceDate'].dt.month
df['MonthName'] = df['InvoiceDate'].dt.strftime('%b')   # Jan, Feb, Mar...
df['DayOfWeek'] = df['InvoiceDate'].dt.day_name()        # Monday, Tuesday...
df['Hour']      = df['InvoiceDate'].dt.hour
df['YearMonth'] = df['InvoiceDate'].dt.to_period('M').astype(str)  # 2010-12, 2011-01...

print("New columns added:")
print(df[['InvoiceDate','Year','Month','MonthName',
          'DayOfWeek','Hour','YearMonth','TotalRevenue']].head(8))

New columns added:
          InvoiceDate  Year  Month MonthName  DayOfWeek  Hour YearMonth  \
0 2010-12-01 08:26:00  2010     12       Dec  Wednesday     8   2010-12   
1 2010-12-01 08:26:00  2010     12       Dec  Wednesday     8   2010-12   
2 2010-12-01 08:26:00  2010     12       Dec  Wednesday     8   2010-12   
3 2010-12-01 08:26:00  2010     12       Dec  Wednesday     8   2010-12   
4 2010-12-01 08:26:00  2010     12       Dec  Wednesday     8   2010-12   
5 2010-12-01 08:26:00  2010     12       Dec  Wednesday     8   2010-12   
6 2010-12-01 08:26:00  2010     12       Dec  Wednesday     8   2010-12   
7 2010-12-01 08:28:00  2010     12       Dec  Wednesday     8   2010-12   

   TotalRevenue  
0         15.30  
1         20.34  
2         22.00  
3         20.34  
4         20.34  
5         15.30  
6         25.50  
7         11.10  


In [19]:
# Save to cleaned_data folder
df.to_csv('../cleaned_data/retail_clean.csv', index=False)

In [20]:
print("File saved: ../cleaned_data/retail_clean.csv")
print(f"Size: {len(df):,} rows × {df.shape[1]} columns")
print("\nAll 14 columns in clean file:")
for col in df.columns:
    print(f"  - {col}")

File saved: ../cleaned_data/retail_clean.csv
Size: 392,692 rows × 15 columns

All 14 columns in clean file:
  - InvoiceNo
  - StockCode
  - Description
  - Quantity
  - InvoiceDate
  - UnitPrice
  - CustomerID
  - Country
  - TotalRevenue
  - Year
  - Month
  - MonthName
  - DayOfWeek
  - Hour
  - YearMonth


In [2]:
import pandas as pd

df = pd.read_csv('../cleaned_data/retail_clean.csv')

In [3]:
# Count orders per customer
customer_orders = df.groupby('CustomerID')['InvoiceNo'].nunique().reset_index()
customer_orders.columns = ['CustomerID', 'OrderCount']

In [4]:
# Label each customer
customer_orders['CustomerType'] = customer_orders['OrderCount'].apply(
    lambda x: 'One-time' if x == 1 else 'Repeat'
)

In [5]:
# Merge back into main dataframe
df = df.merge(customer_orders[['CustomerID', 'CustomerType']], 
              on='CustomerID', how='left')

print(df['CustomerType'].value_counts())
print(f"\nTotal customers: {df['CustomerID'].nunique()}")

CustomerType
Repeat      359794
One-time     32898
Name: count, dtype: int64

Total customers: 4338


In [6]:
# Save updated clean file
df.to_csv('../cleaned_data/retail_clean.csv', index=False)
print("\nFile saved with CustomerType column!")


File saved with CustomerType column!


In [1]:
import pandas as pd
df = pd.read_csv('../cleaned_data/retail_clean.csv')
print(df.columns.tolist())
print(df['CustomerType'].value_counts())

['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'UnitPrice', 'CustomerID', 'Country', 'TotalRevenue', 'Year', 'Month', 'MonthName', 'DayOfWeek', 'Hour', 'YearMonth', 'CustomerType']
CustomerType
Repeat      359794
One-time     32898
Name: count, dtype: int64
